# <center> Homework 123

In [1]:
import tensorflow as tf
from importlib import reload
import tf_preprocessing_layers
import numpy as np
import time 
import pandas as pd

2026-02-08 23:40:54.593567: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-08 23:40:55.328268: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-08 23:40:59.116977: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/zdravko/Machine_Learning_Intern/venv/lib/python3.13/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


## Task 3

да се имплементира класа TextVectorization, така че да работят примерите в книгата. да поддържа всички варианти за output_mode - one_hot, multi_hot, count tf_idf

In [3]:
reload(tf_preprocessing_layers)
from tf_preprocessing_layers import TextVectorization

train_data = ["To be", "!(to be)", "That's the question", "Be, be, be."]
text_vec_layer = TextVectorization()
text_vec_layer.adapt(train_data)
text_vec_layer(tf.constant(["Be good!", "Question: be or be?"]))

2026-02-08 23:41:21.989355: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


<tf.Tensor: shape=(2, 4), dtype=int32, numpy=
array([[2, 1, 0, 0],
       [6, 2, 1, 2]], dtype=int32)>

In [3]:
reload(tf_preprocessing_layers)
from tf_preprocessing_layers import TextVectorization

text_vec_layer = TextVectorization(output_mode="tf_idf")
text_vec_layer.adapt(train_data)
text_vec_layer(tf.constant(["Be good!", "Question: be or be?"]))

# <tf.Tensor: shape=(2, 6), dtype=float32, numpy=
# [0.96725637, 0.6931472 , 0. , 0. , 0. , 0.],
# [0.96725637, 1.3862944 , 0. , 0. , 0. , 1.0986123 ]], dtype=float32)>

<tf.Tensor: shape=(2, 6), dtype=float32, numpy=
array([[0.96725637, 0.6931472 , 0.        , 0.        , 0.        ,
        0.        ],
       [0.96725637, 1.3862944 , 0.        , 0.        , 0.        ,
        1.0986123 ]], dtype=float32)>

In [4]:
reload(tf_preprocessing_layers)
from tf_preprocessing_layers import TextVectorization

text_vec_layer = TextVectorization(output_mode="one_hot")
text_vec_layer.adapt(train_data)
text_vec_layer(tf.constant(["Be good!", "Question: be or be?"]))


# [[[0, 1, 0, 0, 0, 0],
#   [1, 0, 0, 0, 0, 0]], 
  
# [[0, 0, 0, 0, 0, 1],
#  [0, 1, 0, 0, 0, 0],
#  [1, 0, 0, 0, 0, 0],
#  [0, 1, 0, 0, 0, 0]]]>

<tf.Tensor: shape=(2, 4, 6), dtype=float32, numpy=
array([[[0., 1., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.]],

       [[0., 0., 0., 0., 0., 1.],
        [0., 1., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0.]]], dtype=float32)>

In [5]:
reload(tf_preprocessing_layers)
from tf_preprocessing_layers import TextVectorization

text_vec_layer = TextVectorization(output_mode="multi_hot")
text_vec_layer.adapt(train_data)
text_vec_layer(tf.constant(["Be good!", "Question: be or be?"]))

# [[1, 1, 0, 0, 0, 0],
# [1, 1, 0, 0, 0, 1]]

<tf.Tensor: shape=(2, 6), dtype=float32, numpy=
array([[1., 1., 0., 0., 0., 0.],
       [1., 1., 0., 0., 0., 1.]], dtype=float32)>

In [4]:
reload(tf_preprocessing_layers)
from tf_preprocessing_layers import TextVectorization

text_vec_layer = TextVectorization(output_mode="count")
text_vec_layer.adapt(train_data)
text_vec_layer(tf.constant(["Be good!", "Question: be or be?"]))

# [[1, 1, 0, 0, 0, 0],
# [1, 2, 0, 0, 0, 1]]

tf.Tensor(
[[False  True False False]
 [False False  True False]], shape=(2, 4), dtype=bool)


<tf.Tensor: shape=(2, 6), dtype=float32, numpy=
array([[1., 1., 0., 0., 0., 0.],
       [1., 2., 0., 0., 0., 1.]], dtype=float32)>

## Task 4



да се имплементира алтернативна версия на класа Embeding която съдържа Dense слой без активираща функция и bias тегла.

да се сравни бързодействието със оригиналния Embeding слой.

In [21]:
class DenseEmbeding(tf.keras.layers.Layer):
    def __init__(self, input_dim, output_dim, **kwargs):
        super().__init__(**kwargs)
        self.input_dim = input_dim
        self.dense = self.add_weight(shape=(input_dim, output_dim),
                                 initializer="random_uniform")

    def call(self, x):
        x = tf.one_hot(x, self.input_dim)
        return x @ self.dense

In [22]:
tf.random.set_seed(42)
ocean_prox = ["<1H OCEAN", "INLAND", "NEAR OCEAN", "NEAR BAY", "ISLAND"]

str_lookup_layer = tf.keras.layers.StringLookup()
str_lookup_layer.adapt(ocean_prox)

lookup_and_embed = tf.keras.Sequential([
    tf.keras.Input(shape=(1,), dtype=tf.string),
    str_lookup_layer,
    tf.keras.layers.Embedding(input_dim=str_lookup_layer.vocabulary_size(), output_dim=2)
])

s = time.time()
lookup_and_embed(tf.constant([["<1H OCEAN"], ["ISLAND"], ["<1H OCEAN"]]))
org_emb_time = time.time() - s 

lookup_and_embed(tf.constant([["<1H OCEAN"], ["ISLAND"], ["<1H OCEAN"]]))

<tf.Tensor: shape=(3, 1, 2), dtype=float32, numpy=
array([[[ 0.02775445, -0.01977999]],

       [[-0.01132294, -0.01116119]],

       [[ 0.02775445, -0.01977999]]], dtype=float32)>

In [23]:
tf.random.set_seed(42)
ocean_prox = ["<1H OCEAN", "INLAND", "NEAR OCEAN", "NEAR BAY", "ISLAND"]

str_lookup_layer = tf.keras.layers.StringLookup()
str_lookup_layer.adapt(ocean_prox)

lookup_and_embed = tf.keras.Sequential([
    tf.keras.Input(shape=(1,), dtype=tf.string),
    str_lookup_layer,
    DenseEmbeding(input_dim=str_lookup_layer.vocabulary_size(), output_dim=2)
])

s = time.time()
lookup_and_embed(tf.constant([["<1H OCEAN"], ["ISLAND"], ["<1H OCEAN"]]))
dense_emb_time = time.time() - s 

lookup_and_embed(tf.constant([["<1H OCEAN"], ["ISLAND"], ["<1H OCEAN"]]))

<tf.Tensor: shape=(3, 1, 2), dtype=float32, numpy=
array([[[-0.03930056, -0.00623893]],

       [[-0.0364751 ,  0.01310874]],

       [[-0.03930056, -0.00623893]]], dtype=float32)>

In [24]:
pd.DataFrame(
    [
        [org_emb_time],
        [dense_emb_time]
    ],
    ['Embeding', 'DenseEmbeding'],
    ['Time']
)

,Time
Embeding,0.036313
DenseEmbeding,0.013066


## Task 5

да се имплементира алтернативна версия на класа Embeding със някой от алгоритмите за dimentianality reduction

да се сравни бързодействието със оригиналния Embeding слой.

In [ ]:
class PCAEmbeding(tf.keras.layers.Layer):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim

    def adapt(self, data):
        data = tf.one_hot(data, self.input_dim)
        data_centered = data - tf.reduce_mean(data, axis=0, keepdims=True)
        _, _, v = tf.linalg.svd(data_centered)
        self.PC = self.add_weight(shape=(self.input_dim, self.output_dim),
                                        initializer=v[:, :self.output_dim])

    def call(self, x):
        x = tf.one_hot(x, self.input_dim)
        return x @ self.PC

In [12]:
tf.random.set_seed(42)
ocean_prox = ["<1H OCEAN", "INLAND", "NEAR OCEAN", "NEAR BAY", "ISLAND"]

str_lookup_layer = tf.keras.layers.StringLookup()
str_lookup_layer.adapt(ocean_prox)

pca_emb = PCAEmbeding(input_dim=str_lookup_layer.vocabulary_size(), output_dim=2)
pca_emb.adapt(str_lookup_layer(ocean_prox))

lookup_and_embed = tf.keras.Sequential([
    tf.keras.Input(shape=(1,), dtype=tf.string),
    str_lookup_layer,
    pca_emb
])

s = time.time()
lookup_and_embed(tf.constant([["<1H OCEAN"], ["ISLAND"], ["<1H OCEAN"]]))
pca_emb_time = time.time() - s 

lookup_and_embed(tf.constant([["<1H OCEAN"], ["ISLAND"], ["<1H OCEAN"]]))

<tf.Tensor: shape=(3, 1, 2), dtype=float32, numpy=
array([[[-0.22360684,  0.2886751 ]],

       [[-0.22360681, -0.86602545]],

       [[-0.22360684,  0.2886751 ]]], dtype=float32)>

In [27]:
pd.DataFrame(
    [
        [org_emb_time],
        [dense_emb_time],
        [pca_emb_time]
    ],
    ['Embeding', 'DenseEmbeding', 'PCAEmbeding'],
    ['Time']
)

,Time
Embeding,0.036313
DenseEmbeding,0.013066
PCAEmbeding,0.034668
